# Prefill and decode — a revision notebook

This notebook maps the Week 01 code to the two phases inside `model.generate()`. It does not reimplement the Transformer. It helps us observe the boundary between processing the input prompt (prefill) and generating the answer one token at a time (decode).

## The mental model

```text
messages
  ↓ make_inputs()
input_ids + attention_mask
  ↓ model.generate()
PREFILL: process all existing prompt tokens
  ↓ initial KV cache + first-token prediction
DECODE: generate y1, then y2, then y3, ...
  ↓ tokenizer.decode()
answer text
```

Prefill processes the input conversation. Decode produces the output conversation. `use_cache=True` stores the attention keys and values from the prompt and appends the generated tokens' keys and values during decode.

In [ ]:
import platform, threading, time
from queue import Queue
import torch
import transformers
from transformers import AutoModelForCausalLM, AutoTokenizer
from transformers.generation.streamers import BaseStreamer

MODEL_ID = 'Qwen/Qwen2.5-0.5B-Instruct'
DEVICE = 'mps' if torch.backends.mps.is_available() else ('cuda' if torch.cuda.is_available() else 'cpu')
DTYPE = torch.float16 if DEVICE in ('mps', 'cuda') else torch.float32

def sync_device():
    if DEVICE == 'cuda': torch.cuda.synchronize()
    elif DEVICE == 'mps': torch.mps.synchronize()

tokenizer = AutoTokenizer.from_pretrained(MODEL_ID)
model = AutoModelForCausalLM.from_pretrained(MODEL_ID, torch_dtype=DTYPE).to(DEVICE).eval()
print({'model': MODEL_ID, 'device': DEVICE, 'torch': torch.__version__, 'transformers': transformers.__version__})

## 1. What enters prefill?

The prompt is the complete formatted conversation available before generation. It is not the future answer. `add_generation_prompt=True` adds the assistant position where the model should begin generating.

In [ ]:
messages = [{'role': 'user', 'content': 'Explain KV caching in one sentence.'}]
rendered = tokenizer.apply_chat_template(messages, add_generation_prompt=True, tokenize=False)
encoded = tokenizer.apply_chat_template(messages, add_generation_prompt=True, tokenize=True, return_dict=True, return_tensors='pt')

print('formatted prompt:\n', rendered)
print('prompt token count:', encoded['input_ids'].shape[-1])
print('input_ids:', encoded['input_ids'][0].tolist())
print('attention_mask:', encoded['attention_mask'][0].tolist())

At prefill, all of those prompt tokens are sent through the Transformer layers. The model calculates attention information for them and stores their keys and values in the initial KV cache. The last prompt position then produces the probability distribution for the first output token.

In [ ]:
class TokenEventStreamer(BaseStreamer):
    def __init__(self):
        self.queue = Queue()
        self.skip_prompt = True

    def put(self, value):
        if value.ndim == 1: value = value.unsqueeze(0)
        if self.skip_prompt:
            self.skip_prompt = False
            return
        for token_id in value[0].tolist():
            self.queue.put(('token', int(token_id), time.perf_counter()))

    def end(self):
        self.queue.put(('end', None, time.perf_counter()))

    def __iter__(self):
        while True:
            event = self.queue.get()
            if event[0] == 'end': return
            yield event

In [ ]:
def make_inputs(text):
    messages = [{'role': 'user', 'content': text}]
    encoded = tokenizer.apply_chat_template(messages, add_generation_prompt=True, tokenize=True, return_dict=True, return_tensors='pt')
    return {key: value.to(DEVICE) for key, value in encoded.items()}

def observe_generation(text, max_new_tokens=12):
    inputs = make_inputs(text)
    streamer = TokenEventStreamer()
    kwargs = dict(inputs, streamer=streamer, max_new_tokens=max_new_tokens, do_sample=False, use_cache=True)
    sync_device(); started = time.perf_counter()
    thread = threading.Thread(target=model.generate, kwargs=kwargs)
    thread.start()
    events = []
    print('decode tokens: ', end='', flush=True)
    for event in streamer:
        events.append(event)
        token_id, timestamp = event[1], event[2]
        print(repr(tokenizer.decode([token_id], skip_special_tokens=True)), end=' ', flush=True)
    thread.join(); sync_device()
    timestamps = [event[2] for event in events]
    print()
    return {
        'prompt_tokens': int(inputs['input_ids'].shape[-1]),
        'generated_tokens': len(events),
        'ttft_approx_seconds': timestamps[0] - started if timestamps else None,
        'decode_intervals_seconds': [b-a for a, b in zip(timestamps, timestamps[1:])],
        'total_seconds': time.perf_counter() - started,
    }

timing = observe_generation('Explain KV caching in one sentence.', max_new_tokens=12)
timing

## 2. Map the observed timeline

```text
started timer
  │
  ├── prefill: process all prompt tokens
  │             build initial KV cache
  │             predict first output token y1
  │
  ├── TTFT boundary: y1 reaches the streamer
  │
  ├── decode y2: reuse prompt + y1 KV entries
  ├── decode y3: reuse prompt + y1 + y2 KV entries
  └── continue until EOS or max_new_tokens
```

The notebook does not directly expose the model's internal prefill timer. `ttft_approx_seconds` includes prefill plus the time until the first token is delivered to the streamer. The decode intervals approximate the time between generated token events.

## 3. What `use_cache=True` changes

With caching:

```text
prefill: prompt → initial KV cache
y1: reuse prompt cache, append y1 K/V
y2: reuse prompt+y1 cache, append y2 K/V
y3: reuse prompt+y1+y2 cache, append y3 K/V
```

Without caching, each decode step would recompute the prompt and all earlier output tokens. The cache is internal model state for this generation call; it is not the completed text and is not automatically reused for the next HTTP request.

## Revision questions

1. Which code prepares `input_ids`? (`make_inputs`)
2. Which line starts both prefill and decode? (`model.generate(...)`)
3. Which event approximates the end of prefill? (first generated token reaches the streamer)
4. What does `use_cache=True` store? (per-layer attention keys and values)
5. Does the current notebook reuse the cache for a later request? (no)
6. What does `tokenizer.decode()` do? (turns generated token IDs into text)